# Formal SNN Verification Standard Evidence

This notebook exercises SC-NeuroCore's formal SNN verification standard profile and conformance-report arithmetic.

## Evidence Boundary

This notebook verifies the standard's local evidence-accounting behavior only. It does not generate an external theorem-prover proof, model-checker proof, HDL proof, safety certification, or unbounded semantic-correctness claim. Publication-grade use still requires real external proof artefacts and review of their stated assumptions.

In [ ]:
from __future__ import annotations

from pathlib import Path

from sc_neurocore.verification import (
    SNNVerificationEvidence,
    VerificationClaimStatus,
    VerificationEvidenceKind,
    VerificationLevel,
    assess_snn_verification_standard,
    publication_grade_snn_standard_profile,
)

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

profile = publication_grade_snn_standard_profile()
profile_summary = {
    "profile_id": profile.profile_id,
    "requirement_ids": [item.requirement_id for item in profile.requirements],
    "mandatory_requirement_ids": [
        item.requirement_id for item in profile.requirements if item.mandatory
    ],
    "optional_requirement_ids": [
        item.requirement_id for item in profile.requirements if not item.mandatory
    ],
}
assert profile_summary["profile_id"] == "publication-grade-snn-v1"
assert "external_formal_proof" in profile_summary["mandatory_requirement_ids"]
profile_summary

In [ ]:
def evidence(
    evidence_id: str,
    level: VerificationLevel,
    kind: VerificationEvidenceKind,
    status: VerificationClaimStatus = VerificationClaimStatus.PASS,
) -> SNNVerificationEvidence:
    return SNNVerificationEvidence(
        evidence_id=evidence_id,
        level=level,
        kind=kind,
        status=status,
        description=f"{evidence_id} evidence",
        artefact=f"{evidence_id}.json",
        digest="0" * 64,
    )

passing_report = assess_snn_verification_standard(
    (
        evidence("temporal", VerificationLevel.TEMPORAL_PROPERTIES, VerificationEvidenceKind.TEMPORAL_RESULT),
        evidence("interval", VerificationLevel.INTERVAL_PROOF, VerificationEvidenceKind.INTERVAL_BOUND),
        evidence("equiv", VerificationLevel.IMPLEMENTATION_EQUIVALENCE, VerificationEvidenceKind.EQUIVALENCE_TEST),
        evidence("lean", VerificationLevel.EXTERNAL_FORMAL_PROOF, VerificationEvidenceKind.FORMAL_TOOL_LOG),
    )
)
passing_summary = {
    "schema_version": passing_report.schema_version,
    "passed": passing_report.passed,
    "mandatory_coverage": passing_report.mandatory_coverage,
    "missing_mandatory": list(passing_report.missing_mandatory),
    "failed_mandatory": list(passing_report.failed_mandatory),
}
assert passing_summary["passed"] is True
assert passing_summary["mandatory_coverage"] == 1.0
passing_summary

In [ ]:
missing_external_report = assess_snn_verification_standard(
    (
        evidence("temporal", VerificationLevel.TEMPORAL_PROPERTIES, VerificationEvidenceKind.TEMPORAL_RESULT),
        evidence("interval", VerificationLevel.INTERVAL_PROOF, VerificationEvidenceKind.INTERVAL_BOUND),
        evidence("equiv", VerificationLevel.IMPLEMENTATION_EQUIVALENCE, VerificationEvidenceKind.EQUIVALENCE_TEST),
    )
)
missing_external_summary = {
    "passed": missing_external_report.passed,
    "mandatory_coverage": missing_external_report.mandatory_coverage,
    "missing_mandatory": list(missing_external_report.missing_mandatory),
}
assert missing_external_summary["passed"] is False
assert "external_formal_proof" in missing_external_summary["missing_mandatory"]
missing_external_summary

In [ ]:
failed_equivalence_report = assess_snn_verification_standard(
    (
        evidence("temporal", VerificationLevel.TEMPORAL_PROPERTIES, VerificationEvidenceKind.TEMPORAL_RESULT),
        evidence("interval", VerificationLevel.INTERVAL_PROOF, VerificationEvidenceKind.INTERVAL_BOUND),
        evidence(
            "equiv",
            VerificationLevel.IMPLEMENTATION_EQUIVALENCE,
            VerificationEvidenceKind.EQUIVALENCE_TEST,
            VerificationClaimStatus.FAIL,
        ),
        evidence("lean", VerificationLevel.EXTERNAL_FORMAL_PROOF, VerificationEvidenceKind.FORMAL_TOOL_LOG),
    )
)
failed_equivalence_summary = {
    "passed": failed_equivalence_report.passed,
    "failed_mandatory": list(failed_equivalence_report.failed_mandatory),
}
assert failed_equivalence_summary["passed"] is False
assert "implementation_equivalence" in failed_equivalence_summary["failed_mandatory"]
failed_equivalence_summary

In [ ]:
wrong_kind_report = assess_snn_verification_standard(
    (
        evidence("wrong", VerificationLevel.EXTERNAL_FORMAL_PROOF, VerificationEvidenceKind.TRACE),
    )
)
wrong_kind_summary = {
    "passed": wrong_kind_report.passed,
    "missing_mandatory": list(wrong_kind_report.missing_mandatory),
    "matched_wrong_kind_evidence": [item.evidence_id for item in wrong_kind_report.evidence],
}
assert wrong_kind_summary["passed"] is False
assert "external_formal_proof" in wrong_kind_summary["missing_mandatory"]
wrong_kind_summary

In [ ]:
optional_safety_case_report = assess_snn_verification_standard(
    (
        evidence("temporal", VerificationLevel.TEMPORAL_PROPERTIES, VerificationEvidenceKind.TEMPORAL_RESULT),
        evidence("interval", VerificationLevel.INTERVAL_PROOF, VerificationEvidenceKind.INTERVAL_BOUND),
        evidence("equiv", VerificationLevel.IMPLEMENTATION_EQUIVALENCE, VerificationEvidenceKind.EQUIVALENCE_TEST),
        evidence("lean", VerificationLevel.EXTERNAL_FORMAL_PROOF, VerificationEvidenceKind.FORMAL_TOOL_LOG),
        evidence("safety", VerificationLevel.BOUNDED_SIMULATION, VerificationEvidenceKind.SAFETY_CASE),
    )
)
optional_summary = {
    "passed": optional_safety_case_report.passed,
    "mandatory_coverage": optional_safety_case_report.mandatory_coverage,
    "safety_case_result": next(
        item.to_dict()
        for item in optional_safety_case_report.requirement_results
        if item.requirement.requirement_id == "safety_case_traceability"
    ),
}
assert optional_summary["passed"] is True
assert optional_summary["safety_case_result"]["status"] == "pass"
optional_summary

In [ ]:
try:
    SNNVerificationEvidence(
        evidence_id="",
        level=VerificationLevel.INTERVAL_PROOF,
        kind=VerificationEvidenceKind.INTERVAL_BOUND,
        status=VerificationClaimStatus.PASS,
        description="bad",
    )
except ValueError as exc:
    empty_id_refusal = str(exc)
else:
    raise AssertionError("empty evidence id was accepted")

guardrail_summary = {"empty_id_refusal": empty_id_refusal}
assert "evidence_id" in empty_id_refusal
guardrail_summary

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.formal-snn-verification-standard-evidence.v1",
    "profile_summary": profile_summary,
    "passing_summary": passing_summary,
    "missing_external_summary": missing_external_summary,
    "failed_equivalence_summary": failed_equivalence_summary,
    "wrong_kind_summary": wrong_kind_summary,
    "optional_summary": optional_summary,
    "guardrails": guardrail_summary,
    "evidence_boundary": "Local evidence-accounting behavior only; no external prover, model-checker, HDL proof, safety certification, or unbounded semantic-correctness claim.",
}
manifest